<a href="https://colab.research.google.com/github/Kinkyamiee/Diabetes-Prediction/blob/main/VectorStore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Import necessary libraries
import os
import pickle
import numpy as np
import faiss
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.docstore import InMemoryDocstore
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_ollama import OllamaLLM

In [4]:
# Load text file
file_path = "/content/combined_output.txt"  # Replace with your text file path
loader = TextLoader(file_path)
documents = loader.load()

# Split text into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)
docs = text_splitter.split_documents(documents)
print(f"Split documents into {len(docs)} chunks")

# Display the first document chunk
docs[0]

Split documents into 7642 chunks


Document(metadata={'source': '/content/combined_output.txt'}, page_content='the present and future council perspectives exercise at the extremes the amount of exercise to reduce cardiovascular events. abstract habitual physical activity and regular exercise training improve cardiovascular health and longevity. a physically active lifestyle is, therefore, a key aspect of primary and secondary prevention strategies. an appropriate volume and intensity are essential to maximally bene t from exercise interventions. this document summarizes available evidence on the relationship between the exercise volume and risk reductions in cardiovascular morbidity and mortality. furthermore, the risks and benefits of moderate versus high intensity exercise interventions are compared. findings are presented for the general population and cardiac patients eligible for cardiac rehabilitation. finally, the controversy of excessive volumes of exercise in the athletic population is discussed. j am coll card

In [8]:
# Initialize the embedding model without specifying the base_url
embedding_model = OllamaEmbeddings(model="qwen2.5:1.5b")

# Create embeddings for each text chunk
embedded_texts = []
for doc in docs:
    embedding = embedding_model.embed_documents([doc.page_content])[0]
    embedded_texts.append(embedding)

# Convert to numpy array for FAISS
embedded_texts = np.array(embedded_texts, dtype="float32")
print(f"Created embeddings with shape: {embedded_texts.shape}")


ValueError: Error raised by inference endpoint: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/embeddings (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7b0a5a063c50>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [ ]:
# Create FAISS index
dimension = embedded_texts.shape[1]
index = faiss.IndexFlatL2(dimension)  # L2 distance for similarity search
index.add(embedded_texts)  # Add embeddings to the FAISS index

# Create docstore and index mapping
docstore = {str(i): doc for i, doc in enumerate(docs)}
index_to_docstore_id = {i: str(i) for i in range(len(docs))}

# Define save directory
save_dir = r"C:\Users\USER\OneDrive\Desktop\MCP\Vector_Store\qwen2.5-1.5b"
os.makedirs(save_dir, exist_ok=True)

# Save FAISS index
index_path = os.path.join(save_dir, "index.faiss")
faiss.write_index(index, index_path)

# Save metadata (docstore and index_to_docstore_id)
metadata_path = os.path.join(save_dir, "index.pkl")
metadata = {
    "docstore": docstore,
    "index_to_docstore_id": index_to_docstore_id
}
with open(metadata_path, "wb") as f:
    pickle.dump(metadata, f)

print("✅ FAISS index and metadata saved successfully!")


In [ ]:
# Load FAISS index
index = faiss.read_index(index_path)

# Load metadata
with open(metadata_path, "rb") as f:
    saved_data = pickle.load(f)

# Reconstruct FAISS vector store
embedding_function = OllamaEmbeddings(model="qwen2.5:1.5b")

vector_store = FAISS(
    index=index,
    docstore=InMemoryDocstore(saved_data.get("docstore", {})),
    index_to_docstore_id=saved_data.get("index_to_docstore_id", {}),
    embedding_function=embedding_function
)

print("✅ FAISS index and metadata loaded successfully!")


In [ ]:
# Set up the retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

# Initialize the LLM
llm = OllamaLLM(model="qwen2.5:1.5b", temperature=0.5)

# Define a prompt template for the QA system
prompt_template = PromptTemplate.from_template(
    "Use the following context to answer the question concisely and precisely:\n\n"
    "{context}\n\n"
    "Q: {question}\n"
    "A (detailed but concise):"
)

# Create the RetrievalQA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt_template},
)

In [ ]:
# Test the LLM with a simple query
response = llm.invoke("What are the common food for kidney disease patient?")
print("LLM Response:", response)


In [ ]:
def ask_question(question):
    """Function to ask questions to the vector store"""
    try:
        # Use the `query` key as required by the RetrievalQA chain
        response = qa_chain.invoke({"query": question})

        # Extract the answer from the response
        answer = response.get("result", "No answer found.")

        # Get source documents for reference
        source_docs = response.get("source_documents", [])

        # Post-process the response to enforce brevity
        max_words = 500  # Set a word limit for the response
        answer = " ".join(answer.split()[:max_words]) + ("..." if len(answer.split()) > max_words else "")

        return answer, source_docs
    except Exception as e:
        return f"Error: {str(e)}", []


In [ ]:
# Example usage
question = "what are the types of kidney function test?"
answer, source_docs = ask_question(question)

print("Question:", question)
print("\nAnswer:", answer)
print("\nSource Documents:")
for i, doc in enumerate(source_docs[:2]):  # Show only first 2 source documents
    print(f"\nDocument {i+1}:")
    print(doc.page_content[:200] + "...")


In [ ]:
# Example usage
question = "Is diabetes a chronic disease?"
answer, source_docs = ask_question(question)

print("Question:", question)
print("\nAnswer:", answer)
